In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

In [2]:
# 4 samples, 3 classes
logits = torch.tensor([
    [ 2.0,  0.5, -1.0],   # sample 0
    [-0.2,  1.5,  0.3],   # sample 1
    [ 0.1, -0.4,  2.2],   # sample 2
    [ 1.0,  0.8,  0.2],   # sample 3
]).to('cuda:0')

targets = torch.tensor([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 2],
    [1, 0, 0],
]).to('cuda:0')


In [ ]:
gamma = 5
probs = F.softmax(logits, dim=1)
p_t = (probs * targets).sum(dim=1)
# p_t, p_t.shape
torch.log(p_t)
torch.pow((1 - p_t), gamma)

loss = - torch.pow((1 - p_t), gamma) * torch.log(p_t)

tensor([-0.0001, -0.0015, -0.0700, -0.0447])

In [ ]:
class MyFocalLossWithLogits(nn.Module):
    """ 
    My custom implementation of Focal Loss (with logits). Had some help from gpt-5. 
    
    Implements the equation:

        FL(p_t) = −α_t (1 − pₜ)^γ log(p_t)
    
    For the forward method, logits and targets shape is identical. 
    """
    def __init__(self, gamma: float=5.0, reduction: str="mean"):
        super(MyFocalLossWithLogits, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
    
    def get_sample_probabilities(self, logits, targets):
        """ Returns the models estimated probability for the correct class """
        probs = F.softmax(logits, dim=1)
        p_t = (probs * targets).sum(dim=1)
        return p_t
    
    def apply_reduction(self, loss):
        if self.reduction == "mean":
            return loss.mean()
        else:
            return loss.sum()
    
    def forward(self, logits, targets):
        p_t = self.get_sample_probabilities(logits, targets)
        p_t = torch.clamp(p_t, min=1e-7, max=1.0)  
        loss = - ((1 - p_t) ** self.gamma) * torch.log(p_t)
        return self.apply_reduction(loss)

In [ ]:
# Testing BCE
loss = nn.BCEWithLogitsLoss()
pred = torch.randn(3, requires_grad=True)
target = torch.empty(3).random_(2)
# pred, target
output = loss(pred, target)
output
# output.backward()

tensor(0.4915, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)

#### Class-Balanced BCE

In [ ]:
# Modified version of Class Balanced Loss function found here:
#   https://github.com/vandit15/Class-balanced-loss-pytorch/
class ClassBalancedBCEWithLogitsLoss(nn.Module):
    """
    Class-Balanced BCE loss using the formula:
    
    ((1 - beta) / (1 - beta^n_y)) * BCEWithLogitsLoss(logits, targets)
    """
    def __init__(self, beta: float=0.9999):
        super().__init__()
        self.beta = beta


    def get_weights(self, targets):
        batch_size =      targets.shape[0]
        no_of_classes =   targets.shape[1]
        samples_per_cls = targets.sum(dim=0).numpy()
        
        beta_per_cls = np.power(self.beta, samples_per_cls)
        weights = (1.0 - self.beta) / (1.0 - beta_per_cls)
        weights = weights / np.sum(weights) * no_of_classes # normalize mean to 1
        weights = torch.tensor(weights).float()
        
        weights = weights.unsqueeze(0)
        weights = weights.repeat(batch_size, 1) * targets
        weights = weights.sum(1)
        weights = weights.unsqueeze(1)
        weights = weights.repeat(1, no_of_classes)

        return weights


    def forward(self, logits, targets):
        
        weights = self.get_weights(targets)
        
        cb_loss = F.binary_cross_entropy_with_logits(
            input=logits,
            target=targets.float(),
            weight=weights,
        )
        
        return cb_loss

In [5]:
torch.manual_seed(0); np.random.seed(0)
no_of_classes = 3
logits = torch.rand(10, no_of_classes).to('cuda:0').float()
targets = torch.randint(0, 2, size = (10, no_of_classes)).to('cuda:0')
beta = 0.9999
# cb_criterion = ClassBalancedBCEWithLogitsLoss()
# cb_criterion(logits, targets)

In [24]:
""" 
logits:  (N, C)
targets: (N, C)
"""
beta = 0.9999
C = targets.shape[1]
samples_per_cls = targets.sum(dim=0)

beta_per_cls = torch.pow(beta, samples_per_cls)
denom = 1.0 - beta_per_cls
weights = (1.0 - beta) / torch.clamp(denom, min=1e-7)
weights = weights / torch.sum(weights) * C # normalize mean to 1

# # class weights to sample weights
weights = (weights.unsqueeze(0) * targets).sum(dim=1)
weights = weights.unsqueeze(1)

F.binary_cross_entropy_with_logits(
    input=logits,
    target=targets.float(),
    weight=weights,
)
# 0.9816

tensor(0.9816, device='cuda:0')